# Spatial dataset download test (run on Colab, not your laptop)

Goal: pull the second spatial dataset for the scProto rebuttal experiment, using Colab's fast connection instead of a slow home connection.

Two candidates tested here:
- **Part A**: 10x Genomics Xenium Human Breast (12-donor FFPE panel) — test sample `S1_Top` only, ~3.5GB bundle, we extract just the RNA matrix + coordinates + cell types and discard the rest (images etc).
- **Part B**: CRC Xenium dataset (Marteau et al. 2026, BioImage Archive S-BIAD2208) — quick remote structure check of the pre-processed `crca_xenium.h5ad` (27GB) without downloading the whole file.

Run cells top to bottom. Part A's Cell 4 checks are important — read the printed status codes before running the big download cell.

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/data/spatial'
BREAST_DIR = os.path.join(DATA_DIR, 'breast_xenium')
CRC_DIR = os.path.join(DATA_DIR, 'crc_xenium')
os.makedirs(BREAST_DIR, exist_ok=True)
os.makedirs(CRC_DIR, exist_ok=True)
print('Data dir:', DATA_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Data dir: /content/drive/MyDrive/data/spatial


## Part A: 10x Xenium Human Breast — test sample S1_Top

Source page: https://www.10xgenomics.com/datasets/xenium-ffpe-human-breast-biomarkers

Two files exist per sample: `outs.zip` (16.7GB, standard bundle) and `xe_outs.zip` (3.5GB, Explorer bundle) — we use the smaller `xe_outs.zip`.

**Known risk**: this CDN (`cf.10xgenomics.com`) is behind Cloudflare bot-protection. Plain `curl`/`requests` got blocked (403) when tried from a home connection. Cell 4 below checks whether Colab's IP/client gets blocked too — read its output before running Cell 6.

In [10]:
import requests

BASE = 'https://cf.10xgenomics.com/samples/xenium/4.0.0/Human_Breast_Biomarkers_S1_Top'
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36',
    'Referer': 'https://www.10xgenomics.com/datasets/xenium-ffpe-human-breast-biomarkers',
}

def check(url):
    r = requests.head(url, headers=HEADERS, timeout=30, allow_redirects=True)
    print(url.split('/')[-1], '->', r.status_code, r.headers.get('content-length'))
    return r.status_code

statuses = []
for fname in ['cell_groups.csv', 'gene_panel.json', 'xe_outs.zip']:
    statuses.append(check(f'{BASE}/Human_Breast_Biomarkers_S1_Top_{fname}'))

if 403 in statuses:
    print('\n>>> BLOCKED (403). Skip Cell 6, run the Playwright fallback cell (Cell 7) instead.')
else:
    print('\n>>> Not blocked. Proceed to Cell 6 for the direct download.')

Human_Breast_Biomarkers_S1_Top_cell_groups.csv -> 200 6155398
Human_Breast_Biomarkers_S1_Top_gene_panel.json -> 200 None
Human_Breast_Biomarkers_S1_Top_xe_outs.zip -> 200 3509308745

>>> Not blocked. Proceed to Cell 6 for the direct download.


### Cell 5: download helper (used by both the direct and fallback paths)

In [11]:
def download(url, out_path, headers=HEADERS, chunk=8*1024*1024):
    r = requests.get(url, headers=headers, stream=True, timeout=60)
    if r.status_code == 403:
        print('BLOCKED (403) for', url)
        return False
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    done = 0
    with open(out_path, 'wb') as f:
        for part in r.iter_content(chunk_size=chunk):
            f.write(part)
            done += len(part)
            if total:
                print(f'\r{out_path.split("/")[-1]}: {done/1e6:.0f}/{total/1e6:.0f} MB ({100*done/total:.1f}%)', end='')
    print()
    return True

os.makedirs(f'{BREAST_DIR}/S1_Top', exist_ok=True)
download(f'{BASE}/Human_Breast_Biomarkers_S1_Top_cell_groups.csv', f'{BREAST_DIR}/S1_Top/cell_groups.csv')
download(f'{BASE}/Human_Breast_Biomarkers_S1_Top_gene_panel.json', f'{BREAST_DIR}/S1_Top/gene_panel.json')

cell_groups.csv: 6/6 MB (100.0%)



True

### Cell 6: direct download of the 3.5GB bundle (only if Cell 4 showed no 403)

Downloads to local Colab disk first (fast), not straight to Drive — we only copy the small extracted pieces to Drive afterward.

In [12]:
LOCAL_TMP = '/content/xe_outs_S1_Top.zip'
ok = download(f'{BASE}/Human_Breast_Biomarkers_S1_Top_xe_outs.zip', LOCAL_TMP)
if ok:
    print('Downloaded', os.path.getsize(LOCAL_TMP)/1e9, 'GB to', LOCAL_TMP)
else:
    print('Direct download failed/blocked — use Cell 7 (Playwright fallback) instead.')

xe_outs_S1_Top.zip: 3509/3509 MB (100.0%)
Downloaded 3.509308745 GB to /content/xe_outs_S1_Top.zip


### Cell 7: Playwright fallback (only run if Cell 4 or Cell 6 showed 403)

Real headless Chromium passes Cloudflare's bot check where plain `requests` doesn't — confirmed this working from a local machine earlier. Skip this cell entirely if Cell 6 already succeeded.

In [13]:
!pip install -q playwright
!playwright install --with-deps chromium

import asyncio
from playwright.async_api import async_playwright

async def browser_download(url, out_path):
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page()
        try:
            async with page.expect_download(timeout=0) as dl_info:
                try:
                    await page.goto(url, timeout=0)
                except Exception:
                    pass
            download = await dl_info.value
            await download.save_as(out_path)
        finally:
            await browser.close()
    print('saved:', out_path, os.path.getsize(out_path)/1e9, 'GB')

LOCAL_TMP = '/content/xe_outs_S1_Top.zip'
await browser_download(f'{BASE}/Human_Breast_Biomarkers_S1_Top_xe_outs.zip', LOCAL_TMP)

Installing dependencies...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.6 MB]
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,144 kB]
Get:12 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [4,155 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/restricted amd64 Packages [7,479 kB]
Fetched 25.5 MB in 3s (7,

### Cell 8: inspect the bundle contents before extracting

Prints candidate members so we can confirm exact file names (the Explorer bundle's internal layout wasn't verified ahead of time).

In [14]:
import zipfile

KEEP_PATTERNS = ['cell_feature_matrix', 'cells.', 'cell_boundaries', 'gene_panel', 'experiment.xenium', 'metrics_summary']

with zipfile.ZipFile(LOCAL_TMP) as z:
    names = z.namelist()
    print(len(names), 'files in bundle. Full listing:')
    for n in names:
        print(' ', n)
    print('\nMatching KEEP_PATTERNS:')
    for n in names:
        if any(k in n for k in KEEP_PATTERNS):
            print(' ', n)

12 files in bundle. Full listing:
  analysis.zarr.zip
  experiment.xenium
  analysis_summary.html
  cell_feature_matrix.zarr.zip
  transcripts.zarr.zip
  cells.zarr.zip
  gene_panel.json
  morphology_focus/
  morphology_focus/ch0000_dapi.ome.tif
  morphology_focus/ch0001_atp1a1_cd45_e-cadherin.ome.tif
  morphology_focus/ch0002_18s.ome.tif
  morphology_focus/ch0003_alphasma_vimentin.ome.tif

Matching KEEP_PATTERNS:
  experiment.xenium
  cell_feature_matrix.zarr.zip
  cells.zarr.zip
  gene_panel.json


### Cell 9: extract only the small needed pieces to Drive, delete the big zip

Adjust `KEEP_PATTERNS` above first if Cell 8's listing used different names than expected.

In [15]:
out_dir = f'{BREAST_DIR}/S1_Top'
with zipfile.ZipFile(LOCAL_TMP) as z:
    for n in z.namelist():
        if any(k in n for k in KEEP_PATTERNS):
            z.extract(n, out_dir)
            print('extracted', n)

os.remove(LOCAL_TMP)
print('deleted the big zip, freed Colab disk')
print('final contents of', out_dir, ':')
for root, _, files in os.walk(out_dir):
    for fn in files:
        p = os.path.join(root, fn)
        print(' ', p, f'{os.path.getsize(p)/1e6:.1f} MB')

extracted experiment.xenium
extracted cell_feature_matrix.zarr.zip
extracted cells.zarr.zip
extracted gene_panel.json
deleted the big zip, freed Colab disk
final contents of /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top :
  /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top/cell_groups.csv 6.2 MB
  /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top/gene_panel.json 0.2 MB
  /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top/experiment.xenium 0.0 MB
  /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top/cell_feature_matrix.zarr.zip 26.4 MB
  /content/drive/MyDrive/data/spatial/breast_xenium/S1_Top/cells.zarr.zip 216.2 MB


## Part B: CRC Xenium (Marteau et al. 2026) — remote structure check

This one is NOT behind Cloudflare — confirmed plain HTTP range requests work. The file is `crca_xenium.h5ad` (27GB total), but we only read its structure remotely here, not the whole thing.

In [16]:
!pip install -q fsspec aiohttp h5py anndata

import fsspec, h5py

CRC_URL = 'https://ftp.ebi.ac.uk/biostudies/fire/S-BIAD/208/S-BIAD2208/Files/xenium/processed/crca_xenium.h5ad'

of = fsspec.open(CRC_URL, mode='rb', block_size=4*1024*1024)
f = of.open()
h5 = h5py.File(f, 'r')

print('top-level:', list(h5.keys()))
print('obs cols:', list(h5['obs'].keys()))
print('var cols:', list(h5['var'].keys()))
print('obsm keys:', list(h5['obsm'].keys()))
print('X:', h5['X'])
if 'sample' in h5['obs']:
    print('n_obs (via sample field):', h5['obs']['sample'].shape)

top-level: ['X', 'layers', 'obs', 'obsm', 'obsp', 'uns', 'var', 'varm', 'varp']
obs cols: ['BRAF', 'CN', 'KRAS', 'Neutrophil', 'Niche', 'age', 'anno_notes', 'cell_area', 'cell_id', 'cell_labels', 'celltype', 'celltype_sub', 'control_codeword_counts', 'control_probe_counts', 'crca_id', 'crca_patient_id', 'deprecated_codeword_counts', 'genomic_control_counts', 'mean_neighbor_dist', 'msi_status', 'name', 'nucleus_area', 'nucleus_count', 'patient_id', 'region', 'scvi130_nb_leiden_2.5', 'scvi130_nb_leiden_3.8', 'segm_meth', 'segmentation_method', 'sex', 'slide', 'stage', 'tissue_region', 'total_counts', 'transcript_counts', 'transcript_density', 'unassigned_codeword_counts', 'z_level']
var cols: ['_index']
obsm keys: ['delaunayr50', 'nichecompass_umap', 'norm_pca', 'norm_pca_nb_umap', 'scvi', 'scvi130_nb_umap', 'spatial']
X: <HDF5 group "/X" (3 members)>


In [17]:
import numpy as np

def describe_obs_col(h5, col):
    node = h5['obs'][col]
    print('--- obs[' + col + '] ---')
    if isinstance(node, h5py.Group):
        print('  type: categorical group, keys:', list(node.keys()))
        cats = node['categories'][:]
        cats = [c.decode() if isinstance(c, bytes) else c for c in cats]
        print('  n_categories=', len(cats))
        print('  categories:', cats[:40], '...' if len(cats) > 40 else '')
        codes = node['codes'][:5000]
        vals, counts = np.unique(codes[codes >= 0], return_counts=True)
        print('  value counts (first 5000 cells):', dict(zip([cats[v] for v in vals], counts.tolist())))
    else:
        print('  type: plain dataset, dtype=', node.dtype, 'shape=', node.shape)
        sample = node[:5000]
        print('  sample values:', np.unique(sample)[:20])

for col in ['Niche', 'CN', 'tissue_region', 'celltype']:
    describe_obs_col(h5, col)
    print()

print('=== uns top-level keys ===')
print(list(h5['uns'].keys()) if 'uns' in h5 else 'no uns')

for k in h5['uns'].keys():
    if any(s in k.lower() for s in ['cn', 'niche', 'neighbor', 'delaunay']):
        print('candidate uns key:', k)


--- obs[Niche] ---
  type: categorical group, keys: ['categories', 'codes']
  n_categories= 6
  categories: ['Cancer', 'Cancer - fibroblast-enriched', 'Cancer - myeloid-enriched', 'Epithelial niche', 'Neutrophil niche', 'Stromal niche'] 
  value counts (first 5000 cells): {'Cancer': 1081, 'Cancer - fibroblast-enriched': 941, 'Cancer - myeloid-enriched': 536, 'Epithelial niche': 806, 'Neutrophil niche': 44, 'Stromal niche': 1592}

--- obs[CN] ---
  type: categorical group, keys: ['categories', 'codes']
  n_categories= 11
  categories: ['CN1', 'CN2', 'CN3', 'CN4', 'CN5', 'CN6', 'CN7', 'CN8', 'CN9', 'CN10', 'CN11'] 
  value counts (first 5000 cells): {'CN1': 1592, 'CN2': 941, 'CN3': 806, 'CN4': 536, 'CN5': 472, 'CN6': 232, 'CN7': 200, 'CN8': 123, 'CN9': 44, 'CN10': 37, 'CN11': 17}

--- obs[tissue_region] ---
  type: categorical group, keys: ['categories', 'codes']
  n_categories= 3
  categories: ['core', 'margin', 'normal'] 
  value counts (first 5000 cells): {'core': 2523, 'margin': 1617

## Report back

After running, tell me:
1. Did Cell 4 show any 403s?
2. Did Part A finish and what landed in `S1_Top/` (file names + sizes from Cell 9)?
3. Did Part B's structure print correctly, and roughly how long did Cell 4 / the h5ad structure check take (tells us if Colab's connection is meaningfully faster)?